#NAIRR Prompt Engineering for Alignment Tutotiral Notebook


## Tutorial Outline: Training-Free Alignment with DSPy

In this interactive session, we will cover the following topics:

**1. Hosting an LLM:** Deploying a local Large Language Model (LLM) utilizing the vLLM engine within the Google Colab environment.

**2. DSPy Basics:** An introduction to the DSPy programming model for automated prompt engineering and pipeline compilation.

**3. Core Building Blocks:** Programmatically defining system prompts through DSPy's foundational pillars: Signatures, Modules, and Optimizers.

**4. GSM8K Example with MIPROv2:** Compiling and evaluating an optimized reasoning pipeline using the GSM8K dataset.

### Installing Required Packages
Setting up our Python environment with the core frameworks on Google Colab T4 GPU

In [1]:
!uv pip install -U dspy

Using Python 3.12.12 environment at: /usr
Resolved 72 packages in 744ms
Prepared 26 packages in 1.05s
Uninstalled 17 packages in 99ms
Installed 26 packages in 112ms
 + asyncer==0.0.8
 - cachetools==6.2.6
 + cachetools==7.0.5
 - charset-normalizer==3.4.4
 + charset-normalizer==3.4.5
 + colorlog==6.10.1
 + diskcache==5.6.3
 + dspy==3.1.3
 + fastuuid==0.14.0
 - filelock==3.24.3
 + filelock==3.25.1
 - fsspec==2025.3.0
 + fsspec==2026.2.0
 + gepa==0.0.26
 - hf-xet==1.3.1
 + hf-xet==1.3.2
 - huggingface-hub==1.5.0
 + huggingface-hub==1.6.0
 + json-repair==0.58.5
 + litellm==1.82.1
 - numpy==2.0.2
 + numpy==2.4.3
 - openai==2.24.0
 + openai==2.26.0
 + optuna==4.7.0
 - pydantic==2.12.3
 + pydantic==2.12.5
 - pydantic-core==2.41.4
 + pydantic-core==2.41.5
 - python-dotenv==1.2.1
 + python-dotenv==1.2.2
 - regex==2025.11.3
 + regex==2026.2.28
 - requests==2.32.4
 + requests==2.32.5
 - rich==13.9.4
 + rich==14.3.3
 - sqlalchemy==2.0.47
 + sqlalchemy==2.0.48
 - urllib3==2.5.0
 + urllib3==2.6.3
 - 

In [2]:
!uv pip install "vllm==0.7.3" "transformers==4.48.3" --break-system-packages
!uv pip install "huggingface-hub>=0.34.0,<1.0" "tokenizers>=0.21,<0.22" --break-system-packages

Using Python 3.12.12 environment at: /usr
Resolved 137 packages in 485ms
Prepared 53 packages in 51.22s
Uninstalled 24 packages in 1.15s
Installed 53 packages in 595ms
 + airportsdata==20260304
 + astor==0.8.1
 + blake3==1.0.8
 + compressed-tensors==0.9.1
 - cupy-cuda12x==14.0.1
 + cupy-cuda12x==13.6.0
 + depyf==0.18.0
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.14.1
 + fastar==0.8.0
 + fastrlock==0.8.3
 + gguf==0.10.0
 - huggingface-hub==1.6.0
 + huggingface-hub==0.36.2
 + interegular==0.3.3
 - lark==1.3.1
 + lark==1.2.2
 + lm-format-enforcer==0.10.12
 + mistral-common==1.9.1
 + msgspec==0.20.0
 - numpy==2.4.3
 + numpy==1.26.4
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu12==9.10.2.21
 +

Restart the runtime to ensure the newly installed package versions take effect. You can either restart manually or run the os.kill command in the cell below to force a restart.

In [ ]:
import os
os.kill(os.getpid(), 9)

## Host local vllm model on Colab


In the terminal window, type the command line below to host our LLM Qwen2.5-3B for inference:

vllm serve "Qwen/Qwen2.5-3B-Instruct" --dtype=half

In [1]:
import dspy

In [4]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-3B-Instruct","object":"model","created":1773151035,"owned_by":"vllm","root":"Qwen/Qwen2.5-3B-Instruct","parent":null,"max_model_len":32768,"permission":[{"id":"modelperm-8c7908d252514019b97493e3e721e7a3","object":"model_permission","created":1773151035,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

This two-line block below connects DSPy to the locally hosted Qwen model so it can begin sending prompts to it.

The first line creates the language model connection using `dspy.LM(...)`. The argument `'openai/Qwen/Qwen2.5-3B-Instruct'` tells DSPy which model to talk to, using the OpenAI-compatible format since vLLM exposes an OpenAI-style API. The `api_base` points to the local vLLM server running on your machine instead of a remote API. The `api_key` is set to `'dummy'` (or can be anything) because the local vLLM server doesn't require a real key, but the OpenAI client format expects one, so a placeholder is passed. Setting `cache=False` disables response caching, ensuring every call gets a fresh response from the model rather than a stored one — important for accurate evaluation. Setting `temperature=1` allows moderate randomness in the model's outputs, which is useful during optimization so the model can explore different reasoning paths.

The second line, `dspy.configure(lm=lm)`, sets this model as the default for all DSPy operations going forward.

In [5]:
lm = dspy.LM('openai/Qwen/Qwen2.5-3B-Instruct', api_base='http://localhost:8000/v1/', api_key='dummy', cache = False, temperature = 1)
dspy.configure(lm=lm)

We can test our language model by calling it directly with a simple question.

In [6]:
lm("What is 2+2?", temperature = 1)

['2 + 2 equals 4.']

### 2. Importing DSPy and the GSM8K Dataset
We will use the **(GSM8K)** dataset as our primary demonstration of the DSPy function. We import the core DSPy framework alongside the dataset and its strict evaluation metric (`gsm8k_metric`) to measure how well our programmatic prompts improve the model's reasoning capabilities.

In [2]:
import dspy
from dspy.datasets.gsm8k import GSM8K, gsm8k_metric

### Preparing the Training and Evaluation Data

Now we load the GSM8K dataset. We will select a small batch of 100 examples to train (optimize) our prompts, and another 100 examples to evaluate our model's performance.

In [3]:
gsm8k = GSM8K()
gsm8k_trainset = gsm8k.train[:100]
THREADS = 32
kwargs = dict(num_threads=THREADS, display_progress=True, display_table=True)
dspy_program = dspy.ChainOfThought("question -> answer")
evaluate = dspy.Evaluate(devset=gsm8k.test[:100], metric=gsm8k_metric, **kwargs)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

100%|██████████| 1319/1319 [00:00<00:00, 17821.54it/s]


We can inspect what the GSM8k_metric is by looking at the source code.

In [ ]:
import inspect
print(inspect.getsource(gsm8k_metric))

def gsm8k_metric(gold, pred, trace=None):
    return int(parse_integer_answer(str(gold.answer))) == int(parse_integer_answer(str(pred.answer)))



In [ ]:
from dspy.datasets.gsm8k import parse_integer_answer
import inspect
print(inspect.getsource(parse_integer_answer))

def parse_integer_answer(answer, only_first_line=True):
    try:
        if only_first_line:
            answer = answer.strip().split("\n")[0]

        # find the last token that has a number in it
        answer = [token for token in answer.split() if any(c.isdigit() for c in token)][-1]
        answer = answer.split(".")[0]
        answer = "".join([c for c in answer if c.isdigit()])
        answer = int(answer)

    except (ValueError, IndexError):
        answer = 0

    return answer



### Building the DSPy Program: Signatures and Modules

Now we will define how the language model should handle our math problems using DSPy's core building blocks. Instead of writing a long, manual system prompt, we declare it programmatically:

* **The Signature (The "What"):** The shorthand string `"question -> answer"` acts as our strict task definition. It tells the framework exactly what information is going in and what needs to come out.
* **The Module (The "How"):** The reasoning strategy we want the model to use.
    * `dspy.Predict` creates a basic **zero-shot** prompt (it just tries to answer directly).
    * `dspy.ChainOfThought` automatically changes the prompt structure to force the model to reason step-by-step before outputting the final answer.

By passing our Signature into a Module, we create an unoptimized DSPy program that is ready to be evaluated.

In [ ]:
program_zeroshot = dspy.Predict("question -> answer")

In [ ]:
program_CoT = dspy.ChainOfThought("question -> answer")

Now we evaluate our baseline program without any optimization.

In [ ]:
result_zeroshot = evaluate(program_zeroshot)

Average Metric: 17.00 / 100 (17.0%): 100%|██████████| 100/100 [00:55<00:00,  1.79it/s]

2026/03/09 17:50:05 INFO dspy.evaluate.evaluate: Average Metric: 17 / 100 (17.0%)


,question,gold_reasoning,example_answer,pred_answer,gsm8k_metric
0,"Amber, Micah, and Ahito ran 52 miles in total. Amber ran 8 miles. ...",Amber ran <<8=8>>8 miles. Micah ran 3.5 * 8 miles = <<3.5*8=28>>28...,16,"To find out how many miles Ahito ran, we first need to determine h...",✔️ [False]
1,Miguel uses 2 pads of paper a week for his drawing. If there are 3...,Miguel uses 30 x 2 = <<30*2=60>>60 sheets of paper every week. The...,240,"To solve this problem, we need to calculate the total number of sh...",✔️ [False]
2,"At a certain grade level, three-fourths of students have a desktop...",Twenty students represent 1 - 3/4 = 1/4 of the students at that le...,80,Let the total number of students in the grade level be \( x \). Ac...,✔️ [False]
3,Comet Halley orbits the sun every 75 years. Bill's dad saw the Com...,Bill saw the Comet for the second time when he was 30 years * 3= <...,15,Bill was 50 years old when he saw the comet for the first time.,✔️ [False]
4,Tom plants 10 trees a year. Every year he also chops down 2 trees ...,He gets 10-2=<<10-2=8>>8 new trees a year After 10 years he has 8*...,91,"To solve this problem, let's first figure out the net increase in ...",✔️ [False]
...,...,...,...,...,...
95,Jeff’s work is 3 miles away. He walks there and back each day he w...,He has to walk 3*2=<<3*2=6>>6 miles per workday That means he walk...,30,Jeff walks a total of 30 miles if he has to work 5 times a week.,✔️ [False]
96,The ratio of the electric poles and electric wires needed to conne...,The total ratio representing the number of electric poles and wire...,15,"To calculate the total number of electric poles required, we start...",✔️ [False]
97,There are twice as many boys as girls at Dr. Wertz's school. If th...,"There are twice as many boys as girls, so if there are 60 girls, t...",36,"First, we determine the number of boys at Dr. Wertz's school. Sinc...",✔️ [False]
98,Chris is way behind on his math homework. He has 100 math problems...,On Tuesday he completes 3 * 12 = <<3*12=36>>36 math problems. This...,39,"Chris completes 36 problems on Tuesday, which totals to 12 + 36 = ...",✔️ [True]


In [ ]:
result_CoT = evaluate(program_CoT)

Average Metric: 59.00 / 100 (59.0%): 100%|██████████| 100/100 [00:54<00:00,  1.83it/s]

2026/03/09 17:35:48 INFO dspy.evaluate.evaluate: Average Metric: 59 / 100 (59.0%)


,question,gold_reasoning,example_answer,reasoning,pred_answer,gsm8k_metric
0,"Amber, Micah, and Ahito ran 52 miles in total. Amber ran 8 miles. ...",Amber ran <<8=8>>8 miles. Micah ran 3.5 * 8 miles = <<3.5*8=28>>28...,16,"First, let's find out how many miles Micah ran, which is 3.5 times...",\[ \text{Miles Micah ran} = 3.5 \times 8 = 28 \] \[ \text{Total mi...,✔️ [False]
1,Miguel uses 2 pads of paper a week for his drawing. If there are 3...,Miguel uses 30 x 2 = <<30*2=60>>60 sheets of paper every week. The...,240,Miguel uses 2 pads of paper a week. Since there are 30 sheets of p...,240,✔️ [True]
2,"At a certain grade level, three-fourths of students have a desktop...",Twenty students represent 1 - 3/4 = 1/4 of the students at that le...,80,"If three-fourths of the students have a desktop computer at home, ...",\[\n\text{Total students} = 20 \times 4 = 80\n\],✔️ [False]
3,Comet Halley orbits the sun every 75 years. Bill's dad saw the Com...,Bill saw the Comet for the second time when he was 30 years * 3= <...,15,"First, we need to establish how old Bill's father was when he firs...",Bill first saw the comet when he was \(30 + 90 - 75 = 45\) years old.,✔️ [False]
4,Tom plants 10 trees a year. Every year he also chops down 2 trees ...,He gets 10-2=<<10-2=8>>8 new trees a year After 10 years he has 8*...,91,"Tom initially starts with 50 trees. Each year, he adds 10 trees an...",91,✔️ [True]
...,...,...,...,...,...,...
95,Jeff’s work is 3 miles away. He walks there and back each day he w...,He has to walk 3*2=<<3*2=6>>6 miles per workday That means he walk...,30,"Jeff has to walk 3 miles to his work and 3 miles back, which total...",30\nJeff walks a total of 30 miles in 5 days.,✔️ [True]
96,The ratio of the electric poles and electric wires needed to conne...,The total ratio representing the number of electric poles and wire...,15,"Given that the ratio of electric poles to electric wires is 1:3, a...",15,✔️ [True]
97,There are twice as many boys as girls at Dr. Wertz's school. If th...,"There are twice as many boys as girls, so if there are 60 girls, t...",36,"First, we need to find out the total number of boys at the school....",36,✔️ [True]
98,Chris is way behind on his math homework. He has 100 math problems...,On Tuesday he completes 3 * 12 = <<3*12=36>>36 math problems. This...,39,The total number of problems completed on Monday is 12. The number...,39,✔️ [True]



Alternatively, we can define the program using have more control using Python classes Signature.

By defining a custom signature class, we can strictly guide the model's behavior without manually writing a long prompt:
* **The Docstring:** The text right under the class name (`"""You are a math teacher...."""`) acts as the main instruction for the model.
* **Field Descriptions:** We can add specific details and constraints to our inputs and outputs (like telling the model to *only* output an integer).

Once our detailed signature is built, we simply plug it back into our `Predict` and `ChainOfThought` modules to create our new, more strictly defined programs.

In [ ]:
class GSM8KSignature(dspy.Signature):
    """You are a math teacher. Answer the math question with a numerical answer."""

    question: str = dspy.InputField(desc="A math problem")
    answer: str = dspy.OutputField(desc="A single numerical answer")

program_zeroshot2 = dspy.Predict(GSM8KSignature)

Now evaluate the zero_shot program and inspect the system prompt using dspy.inspect_history(n=1)

In [ ]:
result_zeroshot2 = evaluate(program_zeroshot2)

Average Metric: 18.00 / 100 (18.0%): 100%|██████████| 100/100 [00:27<00:00,  3.58it/s]

2026/03/09 17:52:08 INFO dspy.evaluate.evaluate: Average Metric: 18 / 100 (18.0%)


,question,gold_reasoning,example_answer,pred_answer,gsm8k_metric
0,"Amber, Micah, and Ahito ran 52 miles in total. Amber ran 8 miles. ...",Amber ran <<8=8>>8 miles. Micah ran 3.5 * 8 miles = <<3.5*8=28>>28...,16,12.4,✔️ [False]
1,Miguel uses 2 pads of paper a week for his drawing. If there are 3...,Miguel uses 30 x 2 = <<30*2=60>>60 sheets of paper every week. The...,240,120,✔️ [False]
2,"At a certain grade level, three-fourths of students have a desktop...",Twenty students represent 1 - 3/4 = 1/4 of the students at that le...,80,40,✔️ [False]
3,Comet Halley orbits the sun every 75 years. Bill's dad saw the Com...,Bill saw the Comet for the second time when he was 30 years * 3= <...,15,20,✔️ [False]
4,Tom plants 10 trees a year. Every year he also chops down 2 trees ...,He gets 10-2=<<10-2=8>>8 new trees a year After 10 years he has 8*...,91,238,✔️ [False]
...,...,...,...,...,...
95,Jeff’s work is 3 miles away. He walks there and back each day he w...,He has to walk 3*2=<<3*2=6>>6 miles per workday That means he walk...,30,30,✔️ [True]
96,The ratio of the electric poles and electric wires needed to conne...,The total ratio representing the number of electric poles and wire...,15,15,✔️ [True]
97,There are twice as many boys as girls at Dr. Wertz's school. If th...,"There are twice as many boys as girls, so if there are 60 girls, t...",36,4,✔️ [False]
98,Chris is way behind on his math homework. He has 100 math problems...,On Tuesday he completes 3 * 12 = <<3*12=36>>36 math problems. This...,39,58,✔️ [False]


In [ ]:
dspy.inspect_history(n=1)





[2026-03-09T17:52:08.266104]

System message:

Your input fields are:
1. `question` (str): A math problem
Your output fields are:
1. `answer` (str): A single numerical answer
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a math teacher. Answer the math question with a numerical answer.


User message:

[[ ## question ## ]]
Cole wanted to buy new jeans for a dance contest. At the store, he couldn't decide between tattered jeans and jogger jeans. Since the jeans were on sale, he decided to buy them both. The tattered jeans cost $28 while the jogger jeans cost $6 less than the tattered jeans. He saved a total of $6. If he saved 1/3 of the total savings from the jogger jeans and the rest from the tattered jeans, how much more do jogger jeans originally cost than the tattered jeans?

Res

Now we create a new program using the same Signature but with Chain of Thought Module

In [ ]:
program_CoT2 = dspy.ChainOfThought(GSM8KSignature)

In [ ]:
result_CoT2 = evaluate(program_CoT2)

Average Metric: 59.00 / 100 (59.0%): 100%|██████████| 100/100 [00:56<00:00,  1.76it/s]

2026/03/09 17:54:44 INFO dspy.evaluate.evaluate: Average Metric: 59 / 100 (59.0%)


,question,gold_reasoning,example_answer,reasoning,pred_answer,gsm8k_metric
0,"Amber, Micah, and Ahito ran 52 miles in total. Amber ran 8 miles. ...",Amber ran <<8=8>>8 miles. Micah ran 3.5 * 8 miles = <<3.5*8=28>>28...,16,"Amber ran 8 miles, and Micah ran 3.5 times what Amber ran. Therefo...",16,✔️ [True]
1,Miguel uses 2 pads of paper a week for his drawing. If there are 3...,Miguel uses 30 x 2 = <<30*2=60>>60 sheets of paper every week. The...,240,"Miguel uses 2 pads a week, and there are 52 weeks in a year. The t...",3120,✔️ [False]
2,"At a certain grade level, three-fourths of students have a desktop...",Twenty students represent 1 - 3/4 = 1/4 of the students at that le...,80,"If three-fourths of students have a desktop computer, then one-fou...",\[ x = 80 \],✔️ [True]
3,Comet Halley orbits the sun every 75 years. Bill's dad saw the Com...,Bill saw the Comet for the second time when he was 30 years * 3= <...,15,Bill saw the comet the second time at three times the age his fath...,60,✔️ [False]
4,Tom plants 10 trees a year. Every year he also chops down 2 trees ...,He gets 10-2=<<10-2=8>>8 new trees a year After 10 years he has 8*...,91,"Tom starts with 50 trees. Each year, he plants 10 trees and chops ...",91,✔️ [True]
...,...,...,...,...,...,...
95,Jeff’s work is 3 miles away. He walks there and back each day he w...,He has to walk 3*2=<<3*2=6>>6 miles per workday That means he walk...,30,"Jeff walks 3 miles to work and 3 miles back home, which totals 6 m...",30,✔️ [True]
96,The ratio of the electric poles and electric wires needed to conne...,The total ratio representing the number of electric poles and wire...,15,"The ratio of electric poles to electric wires is 1:3, meaning for ...",\[ P = \frac{W}{3} = \frac{45}{3} = 15 \],✔️ [True]
97,There are twice as many boys as girls at Dr. Wertz's school. If th...,"There are twice as many boys as girls, so if there are 60 girls, t...",36,"First, calculate the total number of boys by doubling the number o...",24,✔️ [False]
98,Chris is way behind on his math homework. He has 100 math problems...,On Tuesday he completes 3 * 12 = <<3*12=36>>36 math problems. This...,39,"On Monday, Chris completes 12 problems. On Tuesday, he completes 1...",39\n[[ ## completed ## ## ]],✔️ [True]


In [ ]:
dspy.inspect_history(n=1)





[2026-03-09T17:54:44.407867]

System message:

Your input fields are:
1. `question` (str): A math problem
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): A single numerical answer
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a math teacher. Answer the math question with a numerical answer.


User message:

[[ ## question ## ]]
Cole wanted to buy new jeans for a dance contest. At the store, he couldn't decide between tattered jeans and jogger jeans. Since the jeans were on sale, he decided to buy them both. The tattered jeans cost $28 while the jogger jeans cost $6 less than the tattered jeans. He saved a total of $6. If he saved 1/3 of the total savings from the jogger jeans and the rest from the tattered jeans, how much more do 

### Optimizing with MIPROv2

Now we use DSPy's **MIPROv2** optimizer to automatically find the best prompt for our program. MIPROv2 works in three steps:

1. **Bootstrap** — generates few-shot example candidates from the training set
2. **Propose** — generates candidate instruction variants
3. **Optimize** — uses Bayesian optimization to find the best combination of instructions and few-shot examples

We configure it with `auto="light"` for a faster run, and allow up to 4 bootstrapped and 4 labeled demonstrations per predictor. The optimizer will evaluate different prompt combinations against the `gsm8k_metric` to find the highest-scoring program.

In [ ]:
kwargs = dict(num_threads=32)
optimizer = dspy.MIPROv2(metric=gsm8k_metric, auto="light", **kwargs)

kwargs = dict(max_bootstrapped_demos=4, max_labeled_demos=4)
optimized_dspy_program = optimizer.compile(program_CoT2, trainset=gsm8k.train, **kwargs)

2026/03/09 17:54:44 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 100

2026/03/09 17:54:44 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/03/09 17:54:44 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/03/09 17:54:44 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 18%|█▊        | 7/40 [00:33<02:36,  4.73s/it]


Bootstrapped 4 full traces after 7 examples for up to 1 rounds, amounting to 7 attempts.
Bootstrapping set 4/6


  5%|▌         | 2/40 [00:11<03:38,  5.74s/it]


...

2026/03/09 18:05:28 INFO dspy.evaluate.evaluate: Average Metric: 28 / 35 (80.0%)
2026/03/09 18:05:28 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 80.0 on minibatch of size 35 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 5'].
2026/03/09 18:05:28 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [65.71, 68.57, 74.29, 74.29, 85.71, 65.71, 77.14, 82.86, 80.0]
2026/03/09 18:05:28 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [72.0, 80.0]
2026/03/09 18:05:28 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 80.0
2026/03/09 18:05:28 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/03/09 18:05:28 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 12 / 13 - Minibatch ==



Average Metric: 27.00 / 35 (77.1%): 100%|██████████| 35/35 [00:29<00:00,  1.20it/s]

2026/03/09 18:05:58 INFO dspy.evaluate.evaluate: Average Metric: 27 / 35 (77.1%)
2026/03/09 18:05:58 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 77.14 on minibatch of size 35 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 4'].
2026/03/09 18:05:58 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [65.71, 68.57, 74.29, 74.29, 85.71, 65.71, 77.14, 82.86, 80.0, 77.14]
2026/03/09 18:05:58 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [72.0, 80.0]
2026/03/09 18:05:58 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 80.0
2026/03/09 18:05:58 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/03/09 18:05:58 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 13 / 13 - Full Evaluation =====
2026/03/09 18:05:58 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 80.0) from minibatch trials...



Average Metric: 73.00 / 100 (73.0%): 100%|██████████| 100/100 [01:15<00:00,  1.33it/s]

2026/03/09 18:07:13 INFO dspy.evaluate.evaluate: Average Metric: 73 / 100 (73.0%)
2026/03/09 18:07:13 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [72.0, 80.0, 73.0]
2026/03/09 18:07:13 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 80.0
2026/03/09 18:07:13 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/03/09 18:07:13 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/03/09 18:07:13 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 80.0!


In [ ]:
result_optimized = evaluate(optimized_dspy_program)

Average Metric: 69.00 / 100 (69.0%): 100%|██████████| 100/100 [01:27<00:00,  1.15it/s]

2026/03/09 18:14:53 INFO dspy.evaluate.evaluate: Average Metric: 69 / 100 (69.0%)


,question,gold_reasoning,example_answer,reasoning,pred_answer,gsm8k_metric
0,"Amber, Micah, and Ahito ran 52 miles in total. Amber ran 8 miles. ...",Amber ran <<8=8>>8 miles. Micah ran 3.5 * 8 miles = <<3.5*8=28>>28...,16,"To find out how many miles Ahito ran, the first step is to determi...",``` Total distance run: 52 miles Amber's distance: 8 miles Micah's...,✔️ [False]
1,Miguel uses 2 pads of paper a week for his drawing. If there are 3...,Miguel uses 30 x 2 = <<30*2=60>>60 sheets of paper every week. The...,240,"Miguel uses 2 pads of paper each week, which amounts to \(2 \text{...",240,✔️ [True]
2,"At a certain grade level, three-fourths of students have a desktop...",Twenty students represent 1 - 3/4 = 1/4 of the students at that le...,80,Let \( x \) represent the total number of students at that grade l...,80,✔️ [True]
3,Comet Halley orbits the sun every 75 years. Bill's dad saw the Com...,Bill saw the Comet for the second time when he was 30 years * 3= <...,15,"To determine Bill's age when he saw the comet for the first time, ...",15,✔️ [True]
4,Tom plants 10 trees a year. Every year he also chops down 2 trees ...,He gets 10-2=<<10-2=8>>8 new trees a year After 10 years he has 8*...,91,"Tom starts with 50 trees. Each year, he plants 10 trees but chops ...",91,✔️ [True]
...,...,...,...,...,...,...
95,Jeff’s work is 3 miles away. He walks there and back each day he w...,He has to walk 3*2=<<3*2=6>>6 miles per workday That means he walk...,30,"Jeff walks 3 miles to work and 3 miles back home each day, so he w...",30,✔️ [True]
96,The ratio of the electric poles and electric wires needed to conne...,The total ratio representing the number of electric poles and wire...,15,"Given the ratio of electric poles to electric wires is 1:3, and th...",15,✔️ [True]
97,There are twice as many boys as girls at Dr. Wertz's school. If th...,"There are twice as many boys as girls, so if there are 60 girls, t...",36,"There are twice as many boys as girls, which means there are \(2 \...",36,✔️ [True]
98,Chris is way behind on his math homework. He has 100 math problems...,On Tuesday he completes 3 * 12 = <<3*12=36>>36 math problems. This...,39,"On Monday, Chris completes 12 problems. On Tuesday, he completes 3...",39,✔️ [True]


We can inspect the optimized prompt using `dspy.inspect_history(n=1)`. Notice how the system prompt now includes few-shot examples with full reasoning chains. This is exactly the **Chain-of-Thought Few-Shot** prompting strategy covered in the tutorial slides, but generated and selected automatically by DSPy's optimizer.

In [ ]:
dspy.inspect_history(n=1)





[2026-03-09T18:14:53.569068]

System message:

Your input fields are:
1. `question` (str): A math problem
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): A single numerical answer
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a math teacher. Answer the math question with a numerical answer.


User message:

[[ ## question ## ]]
Sandy's monthly phone bill expense is equal to ten times her age now. In two years, Sandy will be three times as old as Kim. If Kim is currently 10 years old, calculate Sandy's monthly phone bill expense.


Assistant message:

[[ ## reasoning ## ]]
To calculate Sandy's monthly phone bill expense, we need to determine Sandy's current age first. 

Given that in two years, Sandy will be three times as old as K

We can also see the few-shot examples that the optimizer has provided in the optimized system prompt.

In [ ]:
optimized_dspy_program.predict.demos

[Example({'augmented': True, 'question': "Sandy's monthly phone bill expense is equal to ten times her age now. In two years, Sandy will be three times as old as Kim. If Kim is currently 10 years old, calculate Sandy's monthly phone bill expense.", 'reasoning': "To calculate Sandy's monthly phone bill expense, we need to determine Sandy's current age first. \n\nGiven that in two years, Sandy will be three times as old as Kim, and Kim is currently 10 years old, we can express this relationship as:\n\\[ \\text{Sandy's age in two years} = 3 \\times (\\text{Kim's age in two years}) \\]\n\nLet Sandy's current age be \\( S \\). Then her age in two years will be \\( S + 2 \\). Kim's age in two years will be \\( 10 + 2 = 12 \\).\n\nSo, this gives us the equation:\n\\[ S + 2 = 3 \\times 12 \\]\n\\[ S + 2 = 36 \\]\n\\[ S = 34 \\]\n\nSandy is currently 34 years old. Since Sandy's monthly phone bill expense is equal to ten times her age now, we can calculate it as:\n\\[ \\text{Monthly phone bill e

Now we can check the alinged system prompt by using the dspy.inspect_history(n=1) function.

###Save Optimizied Program State

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

This saves the optimized program's state. The MIPROv2 tuned instructions and few-shot demonstrations are saved to a JSON file in Google Drive. Setting `save_program=False` means only the prompt parameters are saved, not the model weights themselves, keeping the file lightweight and portable. You can reload this state later with `dspy.load()` and apply it to the same model without needing to re-run the optimization.

In [ ]:
optimized_dspy_program.save("/content/drive/MyDrive/MIPROv2_qwen25_3b.json", save_program=False)

To reload the optimized program, we first create a new `ChainOfThought` program with the same `GSM8KSignature`, then load the saved state from the JSON file. This restores the tuned instructions and few-shot demonstrations without needing to re-run the optimization process.

In [ ]:
load_qwen25_3b_program = dspy.ChainOfThought(GSM8KSignature)
load_qwen25_3b_program.load("/content/drive/MyDrive/MIPROv2_qwen25_3b.json")

We can also re-evaluate the loaded program, swap in a different dataset, run additional optimizers, or continue building on top of it without starting from scratch.

In [ ]:
evaluate(load_qwen25_3b_program)